# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davionnic/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis (starter CSV):** one row = **one pseudonymized content page** (`content_id`), belonging to one client (`client_id`). Verified: 30,000 unique `content_id`s, zero duplicates — grain holds.

**Lane (locked from ML-02/03):** Refresh / Content Opportunity Scoring → ranking/scoring with proxy label `is_declining_label = (trend_direction == "down")`, primary metric **Precision@50** under **client-holdout**.

**Time window (starter):** every traffic / trend field is aggregated over a **trailing 90-day window ending at export**. Nested comparison windows:
- `*_last_30d` → most recent 30 days of that 90d window
- `*_prev_30d` → days 31–60 back
- earliest ~30 days of the 90d window are **not** in the last/prev pair (observed median `(last+prev)/90d ≈ 0.57`)

**What this contract covers this week:** the bundled starter only (`data/raw/content_refresh_anonymized.csv`). Warehouse tables (`dim_*` / `fact_*`) are named as future join targets + gotchas in section 4 — not queried here.

**Output the analysis hands to a human:** a ranked review queue of pages (priority score + reason codes), not a causal "edit this and rank will rise" claim.

In [1]:
import os
import numpy as np
import pandas as pd

# walk up until we find the repo root (works from work/notebooks/ or Colab)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

RAW = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(RAW), f"missing starter CSV at {os.path.abspath(RAW)}"
df = pd.read_csv(RAW)

print("Working dir:", os.getcwd())
print(f"starter rows={len(df):,}  cols={df.shape[1]}  clients={df['client_id'].nunique()}")
print("lane: Refresh / Content Opportunity Scoring | metric: Precision@50 client-holdout")
print("data scope this week: bundled starter CSV only (no HF warehouse query)")

# Grain check: one row per content_id?
dup = int(df["content_id"].duplicated().sum())
print("\n--- grain ---")
print(f"unique content_id={df['content_id'].nunique():,}  duplicate rows={dup}")
assert dup == 0, "grain broken: content_id not unique"
print("grain holds: one row = one content page")

# Window sanity (no calendar dates on starter — relative windows only)
ratio = (df["impressions_last_30d"] + df["impressions_prev_30d"]) / df["impressions_90d"].replace(0, np.nan)
print("\n--- relative windows (export T0) ---")
print("90d totals: [T0-90, T0] | last_30d: [T0-30, T0] | prev_30d: [T0-60, T0-30]")
print(f"median (last30+prev30)/impressions_90d = {ratio.median():.3f}  (~60/90 days of volume, not full window)")
print(f"content_age_days min/max = {df['content_age_days'].min()} / {df['content_age_days'].max()}  (all >= 90 in this slice)")
print(f"impressions_90d min = {df['impressions_90d'].min()}  (every row has >= 1 impression in this slice)")


Working dir: /workspace/flyrank-ml-internship
starter rows=30,000  cols=44  clients=32
lane: Refresh / Content Opportunity Scoring | metric: Precision@50 client-holdout
data scope this week: bundled starter CSV only (no HF warehouse query)

--- grain ---
unique content_id=30,000  duplicate rows=0
grain holds: one row = one content page

--- relative windows (export T0) ---
90d totals: [T0-90, T0] | last_30d: [T0-30, T0] | prev_30d: [T0-60, T0-30]
median (last30+prev30)/impressions_90d = 0.570  (~60/90 days of volume, not full window)
content_age_days min/max = 90 / 564  (all >= 90 in this slice)
impressions_90d min = 1  (every row has >= 1 impression in this slice)


## 2. Fields: feature / label / context / excluded

Sorted against the starter columns + the prep-step additions the pipeline will create. Buckets match `scripts/ml_utils.py` feature lists where the model actually consumes them.

### Label / proxy (never features)
| Field | Role |
|---|---|
| `trend_direction` | Source of the teaching proxy |
| `trend_pct` | Numeric source of `trend_direction` |
| `is_declining_label` | **Target** = 1 when `trend_direction == "down"` (created in prepare step `01_prepare_features.py`) |

Same-window teaching proxy — **not** a future outcome. Claim mode: decision-support ranking only.

### Context (join / split / read — never model features)
| Field | Why context |
|---|---|
| `content_id`, `client_id` | Pseudonyms: joins + **client-holdout** only |
| `provider_used`, `model_used` | LLM provenance; product/ops metadata, not a ranking signal we want the model to chase |

### Features (knowable at export; safe for the starter ranker)
**Numeric (pipeline list):** `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d`, `days_with_impressions`, `days_with_sessions`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`

**Categorical (pipeline list):** `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`

Related raw totals / tiers (`impressions_90d`, `clicks_90d`, `*_last_30d`, `*_prev_30d`, `age_tier_order`, `char_count_tier`, `has_*`, `measurable_opportunity`, …) are either engineered into the logs above, used for baselines/reason codes, or held as context — they stay out of the leaked label columns.

### Excluded (with why)
| Field / pattern | Why excluded |
|---|---|
| `trend_direction`, `trend_pct` | **Label leakage** — define the proxy |
| Raw client names / URLs / private queries | Privacy — never present in this anonymized slice; keep it that way |
| Blind `fillna(0)` on keyword / word-count blanks | Missingness tracks `content_type` → silent category signal |
| Warehouse query-table last-30 cols against a final-month label | Window overlap leakage (future work; noted in section 4) |

In [2]:
# Field classification check against the starter + pipeline conventions
LABEL_SOURCE = ["trend_direction", "trend_pct"]
CONTEXT_IDS = ["content_id", "client_id"]
EXCLUDED_OPS = ["provider_used", "model_used"]

# Mirror scripts/ml_utils.py (numeric list uses log_* that prep creates)
MODEL_NUMERIC_RAW_OR_DERIVED = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",  # become log_*
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

y = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)
df = df.copy()
df["is_declining_label"] = y

print("--- label / proxy ---")
print("definition: is_declining_label = (trend_direction == 'down')")
print(f"positives: {int(y.sum()):,} / {len(df):,} = {y.mean():.1%}")
print("FORBIDDEN as features:", LABEL_SOURCE)
print("overlap check — any label col in feature lists?",
      set(LABEL_SOURCE) & set(MODEL_NUMERIC_RAW_OR_DERIVED + MODEL_CATEGORICAL))

print("\n--- context ---")
print("IDs (split/join only):", CONTEXT_IDS)
print("ops metadata (excluded from model):", EXCLUDED_OPS)

print("\n--- feature coverage on starter columns ---")
present_num = [c for c in MODEL_NUMERIC_RAW_OR_DERIVED if c in df.columns]
present_cat = [c for c in MODEL_CATEGORICAL if c in df.columns]
print(f"numeric-ish present: {len(present_num)}/{len(MODEL_NUMERIC_RAW_OR_DERIVED)}")
print(f"categorical present: {len(present_cat)}/{len(MODEL_CATEGORICAL)}")
print("categoricals:", present_cat)

# trend_pct formula sanity (dictionary claim)
mask = df["impressions_prev_30d"] > 0
calc = (
    (df.loc[mask, "impressions_last_30d"] - df.loc[mask, "impressions_prev_30d"])
    / df.loc[mask, "impressions_prev_30d"] * 100
).round(1)
match = float((calc == df.loc[mask, "trend_pct"]).mean())
print(f"\ntrend_pct formula match on prev>0 rows: {match:.1%}  (observed)")
print("claim mode: proxy label for decision-support ranking — not a future outcome")


--- label / proxy ---
definition: is_declining_label = (trend_direction == 'down')
positives: 16,262 / 30,000 = 54.2%
FORBIDDEN as features: ['trend_direction', 'trend_pct']
overlap check — any label col in feature lists? set()

--- context ---
IDs (split/join only): ['content_id', 'client_id']
ops metadata (excluded from model): ['provider_used', 'model_used']

--- feature coverage on starter columns ---
numeric-ish present: 18/18
categorical present: 8/8
categoricals: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

trend_pct formula match on prev>0 rows: 99.7%  (observed)
claim mode: proxy label for decision-support ranking — not a future outcome


## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim above is checked below against the starter CSV.

| Claim | Observed |
|---|---|
| Grain = one page | 30,000 rows, 30,000 unique `content_id`, 0 duplicates |
| Clients for holdout | 32 clients; pages/client highly skewed (min 3 … max 7,008) |
| Label base rate | 16,262 / 30,000 = **54.2%** `down` |
| Missingness is patterned | `feedly article` → **100%** blank keyword fields; `word_count` blank on **25.7%** overall |
| Rate scale | `ctr` etc. are ×100 percentages (`0.76` = 0.76%) |
| `avg_position == 0` | **1,205** rows = "no position data", **not** rank #0 — and they currently land in `position_tier == top_3` (gotcha) |
| Windows | last+prev cover ~60 of 90 days; `trend_pct` blank when `impressions_prev_30d == 0` (**3,388** rows) |

In [3]:
print("=== GRAIN ===")
g = df.groupby("content_id").size()
print(f"rows={len(df):,}  unique content_id={g.size:,}  max rows/id={g.max()}")
print("dup probe (should be empty):", int((g > 1).sum()), "content_ids with >1 row")

print("\n=== COUNTS ===")
print(f"clients={df['client_id'].nunique()}")
ppc = df.groupby("client_id").size()
print("pages/client describe:\n", ppc.describe().to_string())
print("\ntrend_direction value counts:")
print(df["trend_direction"].value_counts().to_string())
print("\ncontent_type value counts:")
print(df["content_type"].value_counts().to_string())

print("\n=== MISSING VALUES (overall %) ===")
miss = df.isna().mean().sort_values(ascending=False)
print((miss[miss > 0] * 100).round(2).to_string())

print("\n=== MISSINGNESS BY content_type (keyword + length) ===")
kw_cols = ["search_volume", "competition", "competition_level", "cpc", "main_intent", "word_count"]
rows = []
for ct, gdf in df.groupby("content_type"):
    rows.append({"content_type": ct, "n": len(gdf), **{c: f"{gdf[c].isna().mean():.1%}" for c in kw_cols}})
print(pd.DataFrame(rows).to_string(index=False))
print("note: feedly article has 100% blank keyword context — blind fillna(0) encodes content_type")

print("\n=== WINDOWS / SENTINELS ===")
print(f"trend_pct null (prev_30d==0 path): {df['trend_pct'].isna().sum():,}")
print(f"avg_position==0 (no-data sentinel): {(df['avg_position'] == 0).sum():,}")
print("position_tier when avg_position==0:")
print(df.loc[df["avg_position"] == 0, "position_tier"].value_counts().to_string())
print(f"scroll_rate > 100: {(df['scroll_rate'] > 100).sum():,}   ai_traffic_pct > 100: {(df['ai_traffic_pct'] > 100).sum():,}")
print(f"ctr sample (first 3 non-null): {df['ctr'].dropna().head(3).tolist()}  → remember x100 scale")

print("\n=== CLIENT-HOLDOUT SKETCH (no model fit) ===")
rng = np.random.default_rng(42)
clients = np.array(sorted(df["client_id"].unique()))
n_hold = max(1, int(round(0.20 * len(clients))))
hold_clients = set(rng.choice(clients, size=n_hold, replace=False))
hold_mask = df["client_id"].isin(hold_clients)
print(f"clients total={len(clients)}  holdout clients~20% → {n_hold}")
print(f"holdout rows={hold_mask.sum():,} ({hold_mask.mean():.1%})  train rows={(~hold_mask).sum():,}")
print(f"holdout label rate={df.loc[hold_mask, 'is_declining_label'].mean():.1%}  train={df.loc[~hold_mask, 'is_declining_label'].mean():.1%}")
print("split rule locked: hold out clients, never random rows (matches ML-03 / train script)")


=== GRAIN ===
rows=30,000  unique content_id=30,000  max rows/id=1
dup probe (should be empty): 0 content_ids with >1 row

=== COUNTS ===
clients=32
pages/client describe:
 count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000

trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

content_type value counts:
content_type
keyword article       27207
feedly article         2096
comparison article      697

=== MISSING VALUES (overall %) ===
provider_used        71.46
word_count           25.66
char_count           25.66
char_count_tier      25.66
word_count_tier      25.66
model_used           19.11
trend_pct            11.29
competition_level     8.70
cpc                   8.23
search_volume         8.23
competition           8.23
main_intent           7.91
scroll_rate           0.42

=== MISSINGNESS BY 

## 4. Data limits

What this starter (and, later, the warehouse) **cannot** honestly tell us:

1. **Same-window proxy, not a future outcome.** `is_declining_label` is computed from the same trailing 90d / last-vs-prev 30d windows as the features. Beating Precision@50 on it is measurable decision-support, **not** proof a refresh recovers traffic.
2. **No calendar dates on the starter.** We cannot line up absolute feature vs label windows or check unbalanced panel history here — only relative 90d / 30d fields.
3. **Unbalanced client history (warehouse).** Clients differ in `gsc_data_start` / `ga4_data_start` depth. Prefer per-client windows; a global calendar window will mix long and short histories.
4. **GSC-only early rows (warehouse).** Before `ga4_data_start`, GA4 metrics are zero-filled with `ga4_data_available = FALSE` — and the flag is **three-valued** (TRUE / FALSE / NULL). Filter with `IS TRUE` / `IS NOT TRUE`, never `= FALSE` alone.
5. **Query-table window overlap (warehouse).** `fact_content_query_90d` covers the most recent ~90 days of the snapshot. If a label lives in that final month, last-30 / `impressions_90d` on that table can contain the label period — only prev-30-style fields are safe after window alignment.
6. **Mid-window registration.** Daily fact accrues from registration day; the query table may attribute a full 90d via the current URL map → ~1.5% fact mismatch. Treat `content_total_impressions_90d` as the query table's own denominator; `ANY_VALUE()` repeated per-content context columns, never `SUM()`.
7. **Sentinels & scale traps.** `avg_position == 0` ≠ rank 0; rates are ×100; `scroll_rate` / `ai_traffic_pct` can exceed 100; keyword / length missingness follows `content_type`.
8. **Privacy.** No client names, URLs, or private query strings — keep claims on anonymized ids and aggregates only.

**Entities / joins (mental map for later warehouse work, not executed this week):**
`dim_clients` (client) ⨝ `dim_content` (page) ⨝ `fact_content_daily_performance` (page × day) ; optional ⨝ `fact_content_query_90d` (page × query-hash, fixed 90d). Starter CSV ≈ a pre-joined page-level rollup of the first three for one export snapshot.

In [4]:
# Limits, measured on the starter — warehouse limits stay documented (not queried)
print("--- starter limits (observed) ---")
date_like = [
    c for c in df.columns
    if c.lower() in {"report_date", "gsc_data_start", "ga4_data_start"}
    or c.lower().endswith("_date")
    or c.lower().endswith("_at")
]
msg = date_like if date_like else "(none — relative windows only)"
print("calendar/absolute date columns on starter:", msg)
print("proxy is same-window: label from trend_direction; features include same 90d / 30d fields")
print(f"clients={df['client_id'].nunique()} with highly uneven page counts "
      f"(min={ppc.min()}, median={int(ppc.median())}, max={ppc.max()})")
print("age floor: every page age>=90d — cannot study brand-new URLs in this slice")
print(f"feedly rows with zero keyword context: {(df['content_type']=='feedly article').sum():,}")
print(f"word_count unmeasured: {df['word_count'].isna().sum():,} ({df['word_count'].isna().mean():.1%})")
print(f"avg_position sentinel 0 mis-tiered as top_3: "
      f"{((df['avg_position']==0) & (df['position_tier']=='top_3')).sum():,}")

print("\n--- warehouse gotchas (from docs/data-dictionary.md — not queried this week) ---")
warehouse_gotchas = [
    "unbalanced panel → check dim_clients.gsc_data_start before any window",
    "ga4_data_available is THREE-valued (TRUE/FALSE/NULL) → filter IS TRUE / IS NOT TRUE",
    "fact_content_query_90d window overlaps final snapshot months → align before features",
    "per-content context on query table repeats every row → ANY_VALUE, never SUM",
    "mid-window registration → daily fact vs query 90d denominator can disagree (~1.5%)",
    "_sample table is the FINAL month — do not develop label logic inside it",
]
for i, g in enumerate(warehouse_gotchas, 1):
    print(f"  {i}. {g}")

print("\n--- contract output (one sentence) ---")
print("Hand reviewers a client-holdout-aware ranked queue of pages scored for refresh priority "
      "(Precision@50), with reason codes — decision-support only, not a causal SEO claim.")
print("DONE: starter data contract verified with queries; warehouse joins documented as limits.")


--- starter limits (observed) ---
calendar/absolute date columns on starter: (none — relative windows only)
proxy is same-window: label from trend_direction; features include same 90d / 30d fields
clients=32 with highly uneven page counts (min=3, median=567, max=7008)
age floor: every page age>=90d — cannot study brand-new URLs in this slice
feedly rows with zero keyword context: 2,096
word_count unmeasured: 7,699 (25.7%)
avg_position sentinel 0 mis-tiered as top_3: 1,205

--- warehouse gotchas (from docs/data-dictionary.md — not queried this week) ---
  1. unbalanced panel → check dim_clients.gsc_data_start before any window
  2. ga4_data_available is THREE-valued (TRUE/FALSE/NULL) → filter IS TRUE / IS NOT TRUE
  3. fact_content_query_90d window overlaps final snapshot months → align before features
  4. per-content context on query table repeats every row → ANY_VALUE, never SUM
  5. mid-window registration → daily fact vs query 90d denominator can disagree (~1.5%)
  6. _sample table

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

*(Portal submit is out of scope for this agent run — commit + push only.)*